In [1]:
import torch
import torch.nn as nn

class BinaryClassifier(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2, hidden_layers=[256, 128, 64]):
        super().__init__()

        layerlist = []
        n_in = input_size

        for h in hidden_layers:
            layerlist += [
                nn.Linear(n_in, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),
            ]
            n_in = h

        # Final output layer: 1 logit for binary classification
        layerlist.append(nn.Linear(n_in, 1))

        self.network = nn.Sequential(*layerlist)

        # Better initialization (same as before)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        # shape: (batch_size, 1) → squeeze to (batch_size,)
        logits = self.network(x).squeeze(-1)
        return logits


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 0) Load ALREADY-SCALED binary dataset
df_binary = pd.read_csv("artifacts/final_train_scaled_binary_clustered.csv")

TARGET_COL = "MP_binary_high"

# Exclude non-features
exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}

# Use only numeric feature columns
num_cols = df_binary.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

# Build X/y
X = df_binary[feature_cols].to_numpy(np.float32)
y = df_binary[TARGET_COL].to_numpy(np.float32)

# Stratification labels (cluster)
y_strat = df_binary["Structure_Cluster"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("n features:", len(feature_cols))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

# 1) Fix folds once
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr_idx, val_idx) for tr_idx, val_idx in skf.split(X, y_strat)]
print("Built folds:", len(folds))


X shape: (17625, 202)
y shape: (17625,)
n features: 202
Label counts: {np.float32(0.0): np.int64(16853), np.float32(1.0): np.int64(772)}
Built folds: 10


In [3]:
from sklearn.metrics import accuracy_score
from pathlib import Path

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    

from scipy.special import expit  # stable sigmoid

def save_checkpoint(
    model,
    optimizer,
    epoch,
    train_loss,
    val_loss,
    y_train,
    y_val,
    val_loader,
    fold_idx,
    checkpoint_dir,
    checkpoint_tracking,
    is_final=False,
):
    # Put model in eval mode and collect logits
    model.eval()
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            logits = model(xb).cpu().numpy()
            all_logits.append(logits)

    logits_val = np.concatenate(all_logits)   # shape (N,)
    probs_val  = expit(logits_val)            # stable sigmoid
    preds_val  = (probs_val >= 0.5).astype(int)

    # y_val expected as numpy 0/1
    y_val_np = np.asarray(y_val).astype(int)

    # Classification metric: accuracy
    acc = accuracy_score(y_val_np, preds_val)

    # Filename
    if is_final:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
    else:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"

    checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename

    # Save checkpoint (weights + optimizer + losses + metric)
    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc,
        "fold_idx": fold_idx,
        "is_final": is_final,
    }
    torch.save(checkpoint_data, checkpoint_path_full)

    # Track info for CSV
    checkpoint_info = {
        "Fold": fold_idx,
        "Epoch": epoch,
        "Checkpoint_Filename": checkpoint_filename,
        "Checkpoint_Path": str(checkpoint_path_full),
        "Train_Loss": round(train_loss, 6),
        "Val_Loss": round(val_loss, 6),
        "Accuracy": round(acc, 6),
        "Is_Final": is_final,
    }
    checkpoint_tracking.append(checkpoint_info)

    checkpoint_type = "Final" if is_final else "Regular"
    print(
        f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - "
        f"Val Loss: {val_loss:.4f} | Acc: {acc:.4f}"
    )
    return True


In [4]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


def evaluate_fold(
    trial,
    fold_idx,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    hidden_layers,
    learning_rate,
    batch_size,
    dropout_rate,
    weight_decay,
    max_epochs=10**9,
    patience=30,
    min_delta=0,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_classifier",
    save_every_n_epochs=15,
):

    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu (binary classifier, BCEWithLogitsLoss)")

    # ---- checkpoints ----
    checkpoint_tracking = []
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # ---- tensors (y must be float 0/1) ----
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # ---- model + BCEWithLogitsLoss ----
    # IMPORTANT: BinaryClassifier must output RAW LOGITS (no sigmoid in the final layer)
    model = BinaryClassifier(
        input_size=X_train_scaled.shape[1],
        hidden_layers=hidden_layers,
        dropout_rate=dropout_rate,
    ).to(device)


    # ---- compute pos_weight from TRAIN fold only ----
    # pos_weight = (#negative / #positive)
    pos = y_train_tensor.sum()
    neg = y_train_tensor.numel() - pos

    # avoid divide-by-zero in weird folds (shouldn't happen in stratified CV, but safe)
    # pos_weight must be a tensor (usually shape [1] for binary)
    pos_weight = (neg / (pos + 1e-8)).to(device).view(1)

    print(f"[Fold {fold_idx}] pos={pos.item():.0f} neg={neg.item():.0f} pos_weight={pos_weight.item():.3f}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

    # ---- training loop ----
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            # logits: raw model outputs (any real number)
            logits = model(xb).view(-1)
            yb = yb.view(-1)

            # BCEWithLogitsLoss expects logits directly (NO sigmoid here)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb).view(-1)
                yb = yb.view(-1)

                loss = criterion(logits, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # checkpoints (your existing save_checkpoint() computes sigmoid/acc, so it still works)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader,
                fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            # final checkpoint if we didn't just save one
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                    is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # ---- load best ----
    model.load_state_dict(best_state)
    model.eval()

    # save checkpoint tracking csv
    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_classifier.csv", index=False)

    # ---- final accuracy on val set (best model) ----
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_logits.append(model(xb).view(-1).cpu().numpy())

    logits_val = np.concatenate(all_logits)

    # Convert logits -> probabilities for metrics/analysis
    probs_val = torch.sigmoid(torch.from_numpy(logits_val)).numpy()
    preds_val  = (probs_val >= 0.5).astype(int)

    y_val_np = np.asarray(y_val).astype(int)
    acc = accuracy_score(y_val_np, preds_val)

    return best_val_loss, acc, model, train_losses, val_losses, stop_epoch



import time
import numpy as np
from pathlib import Path

trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Hidden layers (same logic)
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    val_losses = []
    accs = []

    # Run this hyperparameter combo across all folds
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]

        # Put classifier checkpoints in a classifier-specific folder
        trial_checkpoint_root = Path("checkpoints_binary_classifier") / f"trial_{trial.number:03d}"

        best_val_loss, acc, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root,
        )

        val_losses.append(best_val_loss)
        accs.append(acc)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_val_loss = float(np.mean(val_losses))
    avg_acc      = float(np.mean(accs))

    print(f"Trial {trial.number}: Avg Val Loss = {avg_val_loss:.4f} | Avg Acc = {avg_acc:.4f}")

    # Optuna minimizes this
    return avg_val_loss



In [5]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print(best_params)

/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-01-06 22:57:18,463] A new study created in memory with name: no-name-921af2be-8454-49cd-803e-917556b694f4


Fold 0: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_000/fold_0
[Fold 0] pos=695 neg=15167 pos_weight=21.823
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 1.1597 | Acc: 0.7969
[Fold 0] Epoch    1 | Train Loss: 1.3838 | Val Loss: 1.1597 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.8923 | Acc: 0.9144
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.9484 | Acc: 0.9353
[Fold 0] Early stopping at epoch 38 (best Val Loss: 0.8687)
[Fold 0] Final checkpoint saved at epoch 38 - Val Loss: 0.9504 | Acc: 0.9376
Fold 1: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_000/fold_1
[Fold 1] pos=702 neg=15160 pos_weight=21.595
[Fold 1] Regular checkpoint saved at epoch 1 - Val Loss: 1.0831 | Acc: 0.7731
[Fold 1] Epoch    1 | Train Loss: 1.3154 | Val Loss: 1.0831 | ES 0/30
[Fold 1] Regular checkpoint saved at ep

[I 2026-01-06 23:00:29,905] Trial 0 finished with value: 0.8768562294276696 and parameters: {'dropout_rate': 0.270088675155809, 'learning_rate': 0.00010747629842456933, 'weight_decay': 0.00534818489408926, 'batch_size': 32, 'h1': 64}. Best is trial 0 with value: 0.8768562294276696.


[Fold 9] Regular checkpoint saved at epoch 45 - Val Loss: 0.9103 | Acc: 0.9359
[Fold 9] Early stopping at epoch 45 (best Val Loss: 0.8142)
Trial 0 finished in 3.19 minutes
Trial 0: Avg Val Loss = 0.8769 | Avg Acc = 0.9119
Fold 0: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_001/fold_0
[Fold 0] pos=695 neg=15167 pos_weight=21.823
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.9773 | Acc: 0.8094
[Fold 0] Epoch    1 | Train Loss: 1.5753 | Val Loss: 0.9773 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.6128 | Acc: 0.8542
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.5929 | Acc: 0.8746
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.5917 | Acc: 0.8758
[Fold 0] Epoch   50 | Train Loss: 0.6984 | Val Loss: 0.6048 | ES 23/30
[Fold 0] Early stopping at epoch 57 (best Val Loss: 0.5772)
[Fold 0] Final checkpoint saved at epoch 57 - Val Loss: 0.6057 | Acc: 0.8871


[I 2026-01-06 23:10:31,939] Trial 1 finished with value: 0.6460662025958299 and parameters: {'dropout_rate': 0.40242535303030147, 'learning_rate': 0.00011191735560613905, 'weight_decay': 0.0024763866447668386, 'batch_size': 64, 'h1': 256}. Best is trial 1 with value: 0.6460662025958299.


...

[Fold 9] Early stopping at epoch 64 (best Val Loss: 0.6094)
[Fold 9] Final checkpoint saved at epoch 64 - Val Loss: 0.6924 | Acc: 0.9149
Trial 17 finished in 5.86 minutes
Trial 17: Avg Val Loss = 0.6803 | Avg Acc = 0.8656
Fold 0: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_018/fold_0
[Fold 0] pos=695 neg=15167 pos_weight=21.823
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 1.1893 | Acc: 0.6529
[Fold 0] Epoch    1 | Train Loss: 1.5774 | Val Loss: 1.1893 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.6961 | Acc: 0.8174
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.6232 | Acc: 0.8520
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.6045 | Acc: 0.8712
[Fold 0] Epoch   50 | Train Loss: 0.7669 | Val Loss: 0.5861 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.5991 | Acc: 0.8809
[Fold 0] Regular checkpoint saved at epoch 75 - Val Loss: 0

[I 2026-01-07 02:31:24,000] Trial 18 finished with value: 0.659642592232142 and parameters: {'dropout_rate': 0.4547410056538566, 'learning_rate': 7.191339588376876e-05, 'weight_decay': 0.00988120054820694, 'batch_size': 64, 'h1': 256}. Best is trial 13 with value: 0.6417662200118814.


[Fold 9] Early stopping at epoch 77 (best Val Loss: 0.6398)
[Fold 9] Final checkpoint saved at epoch 77 - Val Loss: 0.6721 | Acc: 0.8757
Trial 18 finished in 12.57 minutes
Trial 18: Avg Val Loss = 0.6596 | Avg Acc = 0.8582
Fold 0: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_019/fold_0
[Fold 0] pos=695 neg=15167 pos_weight=21.823
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 1.7406 | Acc: 0.9563
[Fold 0] Epoch    1 | Train Loss: 1.6394 | Val Loss: 1.7406 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 1.6727 | Acc: 0.9648
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 1.5717 | Acc: 0.9637
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 1.9016 | Acc: 0.9643
[Fold 0] Epoch   50 | Train Loss: 1.4815 | Val Loss: 1.5234 | ES 26/30
[Fold 0] Early stopping at epoch 54 (best Val Loss: 1.3085)
[Fold 0] Final checkpoint saved at epoch 54 - Val Loss: 1.6595 | Acc: 0.9643

[I 2026-01-07 02:36:24,389] Trial 19 finished with value: 1.4733768030740992 and parameters: {'dropout_rate': 0.279046415811832, 'learning_rate': 0.0003414617706140082, 'weight_decay': 9.155985924945932e-06, 'batch_size': 16, 'h1': 96}. Best is trial 13 with value: 0.6417662200118814.


[Fold 9] Early stopping at epoch 51 (best Val Loss: 1.3999)
[Fold 9] Final checkpoint saved at epoch 51 - Val Loss: 1.8612 | Acc: 0.9688
Trial 19 finished in 5.01 minutes
Trial 19: Avg Val Loss = 1.4734 | Avg Acc = 0.9557
{'dropout_rate': 0.3897002104320227, 'learning_rate': 6.492811991444491e-05, 'weight_decay': 0.008663647642690531, 'batch_size': 64, 'h1': 256}


In [6]:
print("Best Trial Number:", study.best_trial.number)
print(best_params)

Best Trial Number: 13
{'dropout_rate': 0.3897002104320227, 'learning_rate': 6.492811991444491e-05, 'weight_decay': 0.008663647642690531, 'batch_size': 64, 'h1': 256}


In [7]:
from pathlib import Path
import torch
import pandas as pd

# BASE and artifacts_dir should already be defined (same script as before)
BASE = Path.cwd()  # transfer_learning
artifacts_dir = BASE / "artifacts"

# ---------- Directories for final best models + checkpoints ----------
best_models_dir = artifacts_dir / "binary_weighted_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_binary_weighted_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

print("Best hyperparameters from Optuna:", best_params)

# Helper to derive hidden layers from best_params (same logic as in objective)
def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay, "| batch_size:", batch_size)

final_fold_metrics = []

# ---------- Final training loop for all folds (using `folds`, X, y) ----------
# Assumes you already defined:
#   X, y, folds = [(tr_idx, val_idx), ...] earlier (same as in `objective`)
for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final training for fold {fold_idx} ====================")

    X_train_scaled = X[tr_idx]
    y_train        = y[tr_idx]
    X_val_scaled   = X[val_idx]
    y_val          = y[val_idx]

    # Train this fold with the best hyperparameters
    best_val_loss, acc, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,
        fold_idx=fold_idx,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,
        checkpoint_dir=final_ckpt_dir,   # evaluate_fold will make fold_{k}/ inside
        save_every_n_epochs=15,
    )

    # ---------- Save the final (best) model for this fold ----------
    model_path = best_models_dir / f"binary_weighted_best_fold_{fold_idx}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "fold_idx": fold_idx,
            "best_val_bce": float(best_val_loss),
            "val_accuracy": float(acc),
        },
        model_path,
    )
    print(f"[Fold {fold_idx}] Saved best classifier model to: {model_path}")
    print(f"[Fold {fold_idx}] Val BCE: {best_val_loss:.4f} | Val Acc: {acc:.4f} | Stop epoch: {stop_epoch}")

    # store metrics
    final_fold_metrics.append(
        {
            "Fold": fold_idx,
            "Best_Val_BCE": float(best_val_loss),
            "Val_Accuracy": float(acc),
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

# ---------- Save a summary CSV of all folds ----------
metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "classifier_weighted_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved summary of best classifier weighted models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'dropout_rate': 0.3897002104320227, 'learning_rate': 6.492811991444491e-05, 'weight_decay': 0.008663647642690531, 'batch_size': 64, 'h1': 256}
Using hidden_layers: [256, 128, 64]
dropout: 0.3897002104320227 | lr: 6.492811991444491e-05 | wd: 0.008663647642690531 | batch_size: 64

==================== Final training for fold 0 ====================
Fold 0: Training on cpu (binary classifier, BCEWithLogitsLoss)
Checkpoints will be saved to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/checkpoints_binary_weighted_best/fold_0
[Fold 0] pos=695 neg=15167 pos_weight=21.823
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 1.2033 | Acc: 0.6739
[Fold 0] Epoch    1 | Train Loss: 1.7313 | Val Loss: 1.2033 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.6716 | Acc: 0.8134
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.6086 | Acc: 0.8491
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.5733 | Acc: 0.8729
[Fo

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit

confusion_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Confusion matrix for fold {fold_idx} =====")

    # Load validation datas
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # Load best model
    model_path = best_models_dir / f"binary_weighted_best_fold_{fold_idx}.pt"
    checkpoint = torch.load(model_path, map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=checkpoint["hidden_layers"],
        dropout_rate=checkpoint["dropout_rate"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Predict
    with torch.no_grad():
        logits = model(X_val_tensor).cpu().numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    confusion_matrices.append(cm)

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")




===== Confusion matrix for fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1465  221]
 [  13   64]]
TN=1465, FP=221, FN=13, TP=64

===== Confusion matrix for fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1438  255]
 [  13   57]]
TN=1438, FP=255, FN=13, TP=57

===== Confusion matrix for fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1451  234]
 [   9   69]]
TN=1451, FP=234, FN=9, TP=69

===== Confusion matrix for fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1475  226]
 [  12   50]]
TN=1475, FP=226, FN=12, TP=50

===== Confusion matrix for fold 4 =====
Confusion matrix (rows=true, cols=pred):
[[1445  229]
 [  14   75]]
TN=1445, FP=229, FN=14, TP=75

===== Confusion matrix for fold 5 =====
Confusion matrix (rows=true, cols=pred):
[[1463  224]
 [  14   61]]
TN=1463, FP=224, FN=14, TP=61

===== Confusion matrix for fold 6 =====
Confusion matrix (rows=true, cols=pred):
[[1454  207]
 [  15   86]]
TN=1454, FP=207, FN=15, TP=86

===== Confusion matrix for 

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_32295/3966144648.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_loca

In [9]:
mean_cm = np.mean(confusion_matrices, axis=0)

mean_cm_df = pd.DataFrame(
    mean_cm,
    index=["True Low&Int", "True High"],
    columns=["Pred Low&Int", "Pred High"],
)

print("\n===== Mean confusion matrix across folds =====")
print(mean_cm_df)

# Save to CSV
mean_cm_path = best_models_dir / "binary_mean_confusion_matrix.csv"
mean_cm_df.to_csv(mean_cm_path)
print(f"\n✅ Saved mean confusion matrix to: {mean_cm_path}")



===== Mean confusion matrix across folds =====
              Pred Low&Int  Pred High
True Low&Int        1460.3      225.0
True High             13.1       64.1

✅ Saved mean confusion matrix to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/binary_weighted_best_models/binary_mean_confusion_matrix.csv


In [11]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit  # stable sigmoid

def metrics_from_confusion(cm):
    """
    cm = [[TN, FP],
          [FN, TP]]
    """
    TN, FP, FN, TP = cm.ravel()

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": specificity,
    }

all_fold_metrics = []
all_conf_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\nEvaluating fold {fold_idx}")

    # Load best model for this fold
    ckpt = torch.load(best_models_dir / f"binary_weighted_best_fold_{fold_idx}.pt", map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=ckpt["hidden_layers"],
        dropout_rate=ckpt["dropout_rate"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Validation data
    X_val = torch.tensor(X[val_idx], dtype=torch.float32)
    y_val = y[val_idx].astype(int)

    # Predict
    with torch.no_grad():
        logits = model(X_val).numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_val, preds)
    all_conf_matrices.append(cm)

    # Metrics
    metrics = metrics_from_confusion(cm)
    metrics["Fold"] = fold_idx
    all_fold_metrics.append(metrics)

    print("Confusion matrix:\n", cm)
    print(metrics)



Evaluating fold 0
Confusion matrix:
 [[1465  221]
 [  13   64]]
{'Accuracy': np.float64(0.8672716959727736), 'Precision': np.float64(0.22456140350877193), 'Recall': np.float64(0.8311688311688312), 'F1': np.float64(0.35359116022099446), 'Specificity': np.float64(0.868920521945433), 'Fold': 0}

Evaluating fold 1
Confusion matrix:
 [[1438  255]
 [  13   57]]
{'Accuracy': np.float64(0.8479863868406126), 'Precision': np.float64(0.18269230769230768), 'Recall': np.float64(0.8142857142857143), 'F1': np.float64(0.29842931937172773), 'Specificity': np.float64(0.8493797991730656), 'Fold': 1}

Evaluating fold 2
Confusion matrix:
 [[1451  234]
 [   9   69]]
{'Accuracy': np.float64(0.8621667612024957), 'Precision': np.float64(0.22772277227722773), 'Recall': np.float64(0.8846153846153846), 'F1': np.float64(0.36220472440944884), 'Specificity': np.float64(0.8611275964391691), 'Fold': 2}

Evaluating fold 3
Confusion matrix:
 [[1475  226]
 [  12   50]]
{'Accuracy': np.float64(0.8650028360748724), 'Preci

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_32295/207634363.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_models_dir / f"binary_

In [12]:
metrics_df = pd.DataFrame(all_fold_metrics)

mean_metrics = metrics_df.mean(numeric_only=True)
std_metrics  = metrics_df.std(numeric_only=True)

summary_df = pd.DataFrame({
    "Mean": mean_metrics,
    "Std": std_metrics
})

print("\n=== Cross-validated performance ===")
print(summary_df)

summary_df.to_csv(
    best_models_dir / "binary_undersampling_classifier_confusion_metrics_summary.csv"
)



=== Cross-validated performance ===
                 Mean       Std
Accuracy     0.864909  0.008423
Precision    0.221364  0.033197
Recall       0.827674  0.033332
F1           0.348409  0.042254
Specificity  0.866501  0.009265
Fold         4.500000  3.027650


In [ ]:
####-------------------------------------evaluate_fold () with BCEWithLogitsLoss, without weighted/under/oversampling------------------------####


import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


def evaluate_fold(
    trial,
    fold_idx,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    hidden_layers,
    learning_rate,
    batch_size,
    dropout_rate,
    weight_decay,
    max_epochs=10**9,
    patience=30,
    min_delta=0,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_classifier",
    save_every_n_epochs=15,
):

    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu (binary classifier, BCEWithLogitsLoss)")

    # ---- checkpoints ----
    checkpoint_tracking = []
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # ---- tensors (y must be float 0/1) ----
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # ---- model + BCEWithLogitsLoss ----
    # IMPORTANT: BinaryClassifier must output RAW LOGITS (no sigmoid in the final layer)
    model = BinaryClassifier(
        input_size=X_train_scaled.shape[1],
        hidden_layers=hidden_layers,
        dropout_rate=dropout_rate,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

    # ---- training loop ----
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            # logits: raw model outputs (any real number)
            logits = model(xb).view(-1)
            yb = yb.view(-1)

            # BCEWithLogitsLoss expects logits directly (NO sigmoid here)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb).view(-1)
                yb = yb.view(-1)

                loss = criterion(logits, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # checkpoints (your existing save_checkpoint() computes sigmoid/acc, so it still works)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader,
                fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            # final checkpoint if we didn't just save one
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                    is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # ---- load best ----
    model.load_state_dict(best_state)
    model.eval()

    # save checkpoint tracking csv
    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_classifier.csv", index=False)

    # ---- final accuracy on val set (best model) ----
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_logits.append(model(xb).view(-1).cpu().numpy())

    logits_val = np.concatenate(all_logits)

    # Convert logits -> probabilities for metrics/analysis
    probs_val  = 1 / (1 + np.exp(-logits_val))
    preds_val  = (probs_val >= 0.5).astype(int)

    y_val_np = np.asarray(y_val).astype(int)
    acc = accuracy_score(y_val_np, preds_val)

    return best_val_loss, acc, model, train_losses, val_losses, stop_epoch


import time
import numpy as np
from pathlib import Path

trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Hidden layers (same logic)
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    val_losses = []
    accs = []

    # Run this hyperparameter combo across all folds
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]

        # Put classifier checkpoints in a classifier-specific folder
        trial_checkpoint_root = Path("checkpoints_binary_classifier") / f"trial_{trial.number:03d}"

        best_val_loss, acc, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root,
        )

        val_losses.append(best_val_loss)
        accs.append(acc)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_val_loss = float(np.mean(val_losses))
    avg_acc      = float(np.mean(accs))

    print(f"Trial {trial.number}: Avg Val Loss = {avg_val_loss:.4f} | Avg Acc = {avg_acc:.4f}")

    # Optuna minimizes this
    return avg_val_loss
